In [1]:
import numpy as np
import matplotlib.pyplot as plt
from data.molar_mass import molar_masses as mm
import pandas as pd
import seaborn as sns
import matplotlib

# read filtered data in mol per mol unitd
all_data = pd.read_csv('sample_data_mol_per_mol.csv')
all_samples = all_data['Sample Name'].unique()

# cahnge espanola sample names
change_names = lambda x: x.replace('18-2-', '')

# read data in ppm units
nitric_runs_ppb = pd.read_csv('data/nitric_acid_runs_reduced_ppb.csv')
nitric_runs_ppb['Sample Name'] = nitric_runs_ppb['Sample Name'].apply(change_names)
nitric_runs_ppb = nitric_runs_ppb[nitric_runs_ppb['Sample Name'].isin(all_samples)]




/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

# Rock and diagenetic fluid compsitions
# Primary aragonite depsited in oxic column
oxic_aragonite = dotdict(dict(
    # concentrations, ppm
    C = 120000,
    Ca = 387500,
    Mn = 0.01, 
    Sr = 10000, 
    O = 480000, 
    Mg = 2500, 
    Ce = np.mean([0.011,0.896]),
    I = 6,
    # isotopes
    d13C = 5, # d13C
    d18O = 0, # d18O PDB
    dCe = 0.67, # Ce anomaly
    dI = 0.70900 # I/Ca ratio
))

# Primary aragonite depsited in anoxic column
anoxic_aragonite = dotdict(dict(
    # concentrations, ppm
    C = 120000,
    Ca = 387500,
    Mn = 1, 
    Sr = 10000, 
    O = 480000, 
    Mg = 2500, 
    Ce = np.mean([0.011,0.896]), # same as oxic
    I = 394500*0.1317*(7.8802e-15*mm['I'])/(0.01028*mm['I']),
    d13C = 5, # d13C
    d18O = 0, # d18O PDB
    dCe = 1.025, # Ce anomaly
    dI = 0.70900  # I/Ca ratio 
))

# Oxic seawater
oxic_seawater = dotdict(dict( # concentrations in ppm
  C = 29, 
  Ca = 0.01028 * mm['Ca'] * 1e3,
  Mn = 1e-3,
  Sr = 1e-4, #0.00009 * mm['Sr'] * 1e3,
  O = 889e3,
  Mg = 1295,
  Ce = 1.3132e-11 * mm['Ce'] * 1e3,
  I = 0.375e-6 * mm['I'] * 1e3,
  d13C = -15, ##1.54, # d13C VPDB
  d18O = -30.5, # d18O, VPDB -- -8 per mil SMOW
  dCe = 0.46, # Ce anomaly
  dI = 0.709250 
  ))

# Reducing seawater
reducing_seawater = dotdict(dict( # concentrations in ppm
  C = 29,
  Ca = 0.01028 * mm['Ca'] * 1e3,
  Mn = 0.01,
  Sr = 1e-4, # 0.00009 * mm['Sr'] * 1e3, 
  O = 889e3,
  Mg = 1295,
  Ce = 114e-12 * mm['Ce'] * 1e3, 
  I = 1e-9,
  d13C = -1,# np.mean((-25, 5)), # d13C VPDB
  d18O = -30.5, # d18O, VPDB
  dCe = np.mean((0.8, 1.25)), # Ce anomaly
  dI = 0.709250  # I/Ca
  ))

# Meteric seawater
meteoric_fluid = dotdict(dict( # river and/or gw concentrations in ppm
  C = 10,
  Ca = 0.002*mm['Ca'] * 1e3, 
  Mn = 0.005,
  Sr = 0.06, #2.5e-6*mm['Sr'] * 1e3, 
  O = 889e3,
  Mg = np.mean((0.8, 235)),
  Ce = np.mean((10e-6, 10e-4)),
  I = 2.5091e-7 * mm['I'] * 1e3,
  d13C = -3.245,  # d13C VPDB
  d18O = -37.7434, # d18O, VPDB
  dCe = 1.395, # Ce anomaly
  dI = 0.709250 
  ))

def BannerHanson(rock, fluid):
    '''
    Runs Banner Hanson diagenesis model
    Arguments:
        rock: dict with rock properties
        fluid: doct with diagenetic fluid properties
    Returns:
        rock_evolution: dict with the compositional and isotopic
        evolution of the rock and the fluid
    '''

    # Exchange reaction distribution coefficient (Kd):
    Kd_Sr = 0.025               # Kd Sr-Ca
    Kd_Mn = 5                     # Kd Mn-Ca
    Kd_Ce = np.mean([100,296])
    if fluid is meteoric_fluid:
        Kd_I = 0.0064
    else:
        Kd_I = 0.1317
    Kd_C = 1
    Kd_O = 1


    alpha = 1.001
    alpha_O = 1.0102 # 1.03125 # 1.0330
    alpha_Ce = 1.00

    ##
    # Single element distribution coefficient (D):
    CaCO3 = 400000                  # Ca in CaCO3 (ppm)
    Ca_sw = 400                     # Ca in seawater
    DJ = CaCO3/Ca_sw 

    D_Sr = DJ*Kd_Sr                  # for Sr
    D_Mn = DJ*Kd_Mn                  # for Mn
    D_Ce = DJ*Kd_Ce                  # for Ce
    D_I = DJ*Kd_I 
    D_C = rock.C/fluid.C*Kd_C             # for carbon
    D_O = rock.O/fluid.O*Kd_O         # for oxygen


    F = 0.5 # Weight fraction of fluid

    # Rock evoolution
    re = dotdict(dict(
        Sr = np.zeros(int(1e6)),
        Mn = np.zeros(int(1e6)),
        Ce = np.zeros(int(1e6)),
        I = np.zeros(int(1e6)),
        C = np.zeros(int(1e6)),
        O = np.zeros(int(1e6)),
        d13C = np.zeros(int(1e6)),
        d18O = np.zeros(int(1e6)),
        dCe = np.zeros(int(1e6)),
        N = np.zeros(int(1e6))
    ))

    # initial concentrations
    re['Sr'][0] = rock['Sr']
    re['Mn'][0] = rock['Mn']
    re['Ce'][0] = rock['Ce']
    re['I'][0] = rock['I']
    re['C'][0] = rock['C']
    re['O'][0] = rock['O']
    re['N'][0] = 1

    # rock-fluid system
    rs = dotdict(dict(
        Sr = np.zeros(int(1e6)),
        Mn = np.zeros(int(1e6)),
        Ce = np.zeros(int(1e6)),
        I = np.zeros(int(1e6)),
        C = np.zeros(int(1e6)),
        O = np.zeros(int(1e6)),
        d13C = np.zeros(int(1e6)),
        d18O = np.zeros(int(1e6)),
        dCe = np.zeros(int(1e6)),
    ))

    # fluid
    Cf_Sr = np.zeros(2)
    Cf_Sr[0] = fluid['Sr']
    Cf_Mn = np.zeros(2)
    Cf_Mn[0] = fluid['Mn']
    Cf_Ce = np.zeros(2)
    Cf_Ce[0] = fluid['Ce']
    Cf_I = np.zeros(2)
    Cf_I[0] = fluid['I']
    Cf_C = np.zeros(2)
    Cf_C[0] = fluid['C']
    Cf_O = np.zeros(2)
    Cf_O[0] = fluid['O']
    d13C_f = fluid['d13C']
    d18O_f = fluid['d18O']
    dCe_f = fluid['dCe']


    rs.Sr[0] = F*Cf_Sr[0] + (1 - F)*re.Sr[0] 
    rs.Mn[0] = F*Cf_Mn[0] + (1 - F)*re.Mn[0] 
    rs.Ce[0] = F*Cf_Ce[0] + (1 - F)*re.Ce[0] 
    rs.I[0] = F*Cf_I[0] + (1 - F)*re.I[0] 
    rs.C[0] = F*Cf_C[0] + (1 - F)*re.C[0] 
    rs.O[0] = F*Cf_O[0] + (1 - F)*re.O[0]

    # Initial isotopes of the system:
    re.d13C[0] = rock['d13C']
    re.d18O[0] = rock['d18O']
    re.dCe[0] = rock['dCe']

    rs.d13C[0] = (d13C_f*Cf_C[0]*F + re.d13C[0]*re.C[0]*(1 - F))/rs.C[0] 
    rs.d18O[0] = (d18O_f*Cf_O[0]*F + re.d18O[0]*re.O[0]*(1 - F))/rs.O[0] 
    rs.dCe[0]  = (dCe_f*Cf_Ce[0]*F + re.dCe[0]*re.Ce[0]*(1 - F))/rs.Ce[0] 

    ##
    # Closed-system equilibration:

    # Partitioning Sr into the rock:
    re.Sr[1] = rs.Sr[0]/(F/D_Sr + (1 - F)) 
    re.Mn[1] = rs.Mn[0]/(F/D_Mn + (1 - F)) 
    re.Ce[1] = rs.Ce[0]/(F/D_Ce + (1 - F)) 
    re.I[1] = rs.I[0]/(F/D_I + (1 - F)) 
    re.C[1] = rs.C[0]/(F/D_C + (1 - F)) 
    re.O[1] = rs.O[0]/(F/D_O + (1 - F)) 

    # Partitioning into the fluid:
    Cf_Sr[1] = re.Sr[1]/D_Sr 
    Cf_Mn[1] = re.Mn[1]/D_Mn
    Cf_Ce[1] = re.Ce[1]/D_Ce 
    Cf_I[1] = re.I[1]/D_I 
    Cf_C[1] = re.C[1]/D_C 
    Cf_O[1] = re.O[1]/D_O 

    # Check for mass balance:
    rs.Sr[1] = F*Cf_Sr[1] + (1 - F)*re.Sr[1] 
    rs.Mn[1] = F*Cf_Mn[1] + (1 - F)*re.Mn[1] 
    rs.Ce[1] = F*Cf_Ce[1] + (1 - F)*re.Ce[1] 
    rs.I[1] = F*Cf_I[1] + (1 - F)*re.I[1] 
    rs.C[1] = F*Cf_C[1] + (1 - F)*re.C[1] 
    rs.O[1] = F*Cf_O[1] + (1 - F)*re.O[1] 

    # step 2 isotopes of the system:
    re.d13C[1] = (rs.d13C[0]*rs.C[1] - F*d13C_f*Cf_C[1])/((1 - F)*re.C[1]) 
    rs.d13C[1] = (d13C_f*Cf_C[1]*F + re.d13C[1]*re.C[1]*(1 - F))/rs.C[1] 

    re.d18O[1] = (rs.d18O[0]*rs.O[1] - F*d18O_f*Cf_O[1])/((1 - F)*re.O[1]) 
    rs.d18O[1] = (d18O_f*Cf_O[1]*F + re.d18O[1]*re.O[1]*(1 - F))/rs.O[1] 

    re.dCe[1] = (rs.dCe[0]*rs.Ce[1] - F*dCe_f*Cf_Ce[1])/((1 - F)*re.Ce[1]) 
    rs.dCe[1] = (dCe_f*Cf_Ce[1]*F + re.dCe[1]*re.Ce[1]*(1 - F))/rs.Ce[1] 


    ##

    # Open-system equilibration (Cf_I does not change!):
    for i in range(1, int(1e6)):

        # partitioning into the rock:
        re.Sr[i] = rs.Sr[i-1]/(F/D_Sr + (1 - F)) 
        re.Mn[i] = rs.Mn[i-1]/(F/D_Mn + (1 - F))
        re.Ce[i] = rs.Ce[i-1]/(F/D_Ce + (1 - F)) 
        re.I[i] = rs.I[i-1]/(F/D_I + (1 - F)) 
        re.C[i] = rs.C[i-1]/(F/D_C + (1 - F)) 
        re.O[i] = rs.O[i-1]/(F/D_O + (1 - F)) 

        # calculate new mass of system:
        rs.Sr[i] = F*Cf_Sr[0] + (1 - F)*re.Sr[i] 
        rs.Mn[i] = F*Cf_Mn[0] + (1 - F)*re.Mn[i]
        rs.Ce[i] = F*Cf_Ce[0] + (1 - F)*re.Ce[i] 
        rs.I[i] = F*Cf_I[0] + (1 - F)*re.I[i] 
        rs.C[i] = F*Cf_C[0] + (1 - F)*re.C[i] 
        rs.O[i] = F*Cf_O[0] + (1 - F)*re.O[i] 

        # isotopes:
        #rs.d13C[i] = rs.d13C[i-1]                       # no fractionations
        #rs.d18O[i] = rs.d18O[i-1] 

        # with fractionation
        re.d13C[i] = (rs.d13C[i-1]*rs.C[i]*alpha - F*Cf_C[1]*1000*(1-alpha))/\
            (F*Cf_C[1] + alpha*(1-F)*re.C[i]) 

        rs.d13C[i] = (d13C_f*Cf_C[1]*F + re.d13C[i]*re.C[i]*(1 - F))/rs.C[i] 


        re.d18O[i] = (rs.d18O[i-1]*rs.O[i]*alpha_O - F*Cf_O[1]*1000*(1-alpha_O))/\
            (F*Cf_O[1] + alpha_O*(1-F)*re.O[i]) 

        rs.d18O[i] = (d18O_f*Cf_O[1]*F + re.d18O[i]*re.O[i]*(1 - F))/rs.O[i]

        re.dCe[i] = (rs.dCe[i-1]*rs.Ce[i]*alpha_Ce - F*Cf_Ce[1]*1000*(1-alpha_Ce))/\
            (F*Cf_Ce[1] + alpha_Ce*(1-F)*re.Ce[i]) 
    
        rs.dCe[i] = (dCe_f*Cf_Ce[1]*F + re.dCe[i]*re.Ce[i]*(1 - F))/rs.Ce[i] 

        # cummulative fluid:rock ratio
        re.N[i] = i*F/(1-F) 
        
    return re, rs


re_mtfl, rs_mtfl = BannerHanson(rock=oxic_aragonite, fluid=meteoric_fluid)
re_oxsw, rs_oxsw = BannerHanson(rock=oxic_aragonite, fluid=oxic_seawater)
re_rdsw, rs_rdsw = BannerHanson(rock=oxic_aragonite, fluid=reducing_seawater)



In [ ]:
# plot mode results
a4_x, a4_y = (8.27, 11.69)
matplotlib.rc('xtick', labelsize=8) 
matplotlib.rc('ytick', labelsize=8)

def axvspan_with_lines(x1, x2, alpha, color):
    plt.axvspan(x1, x2, alpha=alpha, color=color)
    plt.axvline(x1,color=color, lw=0.8)
    plt.axvline(x2,color=color, lw=0.8)

fig, axs = plt.subplots(6,2 , figsize=[a4_x*0.6,a4_y*0.7],
    gridspec_kw={'width_ratios': [3.5, 1]}, sharey='row')

plt.sca(axs[0][0])
plt.plot(re_mtfl.N,re_mtfl.Sr*1e3, color='#CF8F27', alpha=0.5, label='Meteoric fluid')
plt.plot(re_oxsw.N,re_oxsw.Sr*1e3, color='#3396CC', alpha=0.5, label='Oxidizing seawater')
plt.plot(re_rdsw.N,re_rdsw.Sr*1e3, color='k', alpha=0.5, label='Reducing seawater')
axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
plt.legend(fontsize=7, frameon=False, loc='upper left', ncol=3, 
           bbox_to_anchor=(0, 1.4))

plt.ylabel('Sr (ppb)', fontsize=7)
plt.xscale('log')
plt.yscale('log')

plt.sca(axs[0][1])
sns.kdeplot(nitric_runs_ppb[nitric_runs_ppb['88  Sr  [ No Gas ]']>0],
    y='88  Sr  [ No Gas ]', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=1)
# sns.move_legend(axs[0][1], "upper left", fontsize=7, frameon=0)
legend = axs[0][1].get_legend()
handles = legend.legend_handles
axs[0][1].legend(handles, ['Espanola Fm.', 'Gordon Lake Fm\nKona dolomite', 
    'Mosher Fm.',], title='', fontsize=7, frameon=False, loc="upper left", 
    bbox_to_anchor=(1, 1))
axs[0][1].set(xlabel=None, ylabel=None)


plt.sca(axs[1][0])
plt.plot(re_mtfl.N,re_mtfl.Mn*1e3,color='#CF8F27', alpha=0.5)
plt.plot(re_oxsw.N,re_oxsw.Mn*1e3, color='#3396CC', alpha=0.5)
plt.plot(re_rdsw.N,re_rdsw.Mn*1e3, color='k', alpha=0.5)

axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
plt.ylabel('Mn (ppb)', fontsize=7)
plt.xscale('log')
plt.yscale('log')
# plt.ylim(1e-2, 1e5)

plt.sca(axs[1][1])
sns.kdeplot(nitric_runs_ppb[nitric_runs_ppb['55 -> 55  Mn  [ H2 ]']>0],
    y='55 -> 55  Mn  [ H2 ]', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=0)
axs[1][1].set(xlabel=None, ylabel=None)


plt.sca(axs[2][0])
plt.plot(re_mtfl.N,re_mtfl.dCe, color='#CF8F27', alpha=0.5)
plt.plot(re_oxsw.N,re_oxsw.dCe, color='#3396CC', alpha=0.5)
plt.plot(re_rdsw.N,re_rdsw.dCe, color='k', alpha=0.5)
axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
plt.axhspan(0.8, 1.2, alpha=0.1, color='gray', label='No Ce anomaly')
plt.ylabel(r'Ce/Ce*', fontsize=7)
plt.xscale('log')
plt.legend(fontsize=7, frameon=False)
plt.ylim(0.4, 1.7)

plt.sca(axs[2][1])
sns.kdeplot(all_data,
    y='Ce_anomaly_geom', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=0)
axs[2][1].set(xlabel=None, ylabel=None)

plt.sca(axs[3][0])
plt.plot(re_mtfl.N,re_mtfl.I*1e3, color='#CF8F27', alpha=0.5)
plt.plot(re_oxsw.N,re_oxsw.I*1e3, color='#3396CC', alpha=0.5)
plt.plot(re_rdsw.N,re_rdsw.I*1e3, color='k', alpha=0.5)
axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
plt.ylabel('I (ppb)', fontsize=7)
plt.axhspan(6e-3, 1e-2, alpha=0.1, color='gray', label='I detection limit')
plt.legend(fontsize=7, frameon=False)
plt.xscale('log')
plt.yscale('log')

plt.sca(axs[3][1])
sns.kdeplot(nitric_runs_ppb[nitric_runs_ppb['127  I  [ No Gas ]']>0],
    y='127  I  [ No Gas ]', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=0)
axs[3][1].set(xlabel=None, ylabel=None)

plt.sca(axs[4][0])
plt.plot(re_mtfl.N,re_mtfl.d13C, color='#CF8F27', alpha=0.5)
plt.plot(re_oxsw.N,re_oxsw.d13C, color='#3396CC', alpha=0.5)
plt.plot(re_rdsw.N,re_rdsw.d13C, color='k', alpha=0.5)
axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
plt.ylabel(r'$\delta^{13}C$ (‰)', fontsize=7)
plt.xscale('log')


plt.sca(axs[4][1])
sns.kdeplot(all_data,
    y='d13c_sp', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=0)
axs[4][1].set(xlabel=None, ylabel=None)

plt.sca(axs[5][0])
plt.plot(re_mtfl.N,re_mtfl.d18O, color='#CF8F27', alpha=0.5)
plt.plot(re_oxsw.N,re_oxsw.d18O, color='#3396CC', alpha=0.5)
plt.plot(re_rdsw.N,re_rdsw.d18O, color='k', alpha=0.5)
axvspan_with_lines(200, 600, alpha=0.1, color='green')
axvspan_with_lines(300, 2000, alpha=0.1, color='red')
axvspan_with_lines(300, 4000, alpha=0.1, color='blue')
# plt.scatter((N[0],N[-1]),[-30,-30])
plt.ylabel(r'$\delta^{18}O$ (‰)', fontsize=7)
plt.xlabel('Cumulative fluid to rock ratio', fontsize=8)
plt.xscale('log')


plt.sca(axs[5][1])
sns.kdeplot(all_data,
    y='d18o_sp', 
    hue='formation', fill=True, 
    palette=['b', 'r', 'g'], legend=0)
axs[5][1].set(xlabel=None, ylabel=None)
plt.xlabel('Density', fontsize=8)

plt.subplots_adjust(wspace=0.1, hspace=0.5)
plt.savefig('plots/BannerHanson_model.pdf',  bbox_inches='tight')